In [ ]:
import matplotlib.pyplot as plt
import time
import numpy as np
import torch
from torch import nn, optim, autograd
from math import pi
from sklearn.model_selection import RepeatedKFold


torch.manual_seed(123456)
np.random.seed(123456)

class Unit(nn.Module):
    def __init__(self, in_N, out_N):
        super(Unit, self).__init__()
        self.in_N = in_N
        self.out_N = out_N
        self.L = nn.Linear(in_N, out_N)

    def forward(self, x):
        x1 = self.L(x)
        x2 = torch.tanh(x1)
        return x2

class Unit1(nn.Module):
    def __init__(self, in_N, out_N):
        super(Unit1, self).__init__()
        self.in_N = in_N
        self.out_N = out_N
        self.L = nn.Linear(in_N, out_N)

    def forward(self, x):
        x1 = self.L(x)
        x2 = torch.sigmoid(x1)
        return x2


class NN1(nn.Module): #Nonlinear component
    def __init__(self, in_N, width, depth, out_N):
        super(NN1, self).__init__()
        self.width = width
        self.in_N = in_N
        self.out_N = out_N
        self.stack = nn.ModuleList()
        self.stack.append(Unit(in_N, width[0]))
        for i in range(1,depth-1):
            self.stack.append(Unit(width[i-1], width[i]))
        self.stack.append(Unit1(width[depth-2], width[depth-1])) #nn.Linear

    def forward(self, x):
        for i in range(len(self.stack)):
            x = self.stack[i](x)
        return x

    
class NN2(nn.Module): #Linear component
    def __init__(self, in_N, width, depth, out_N):
        super(NN2, self).__init__()
        self.in_N = in_N
        self.width = width
        self.depth = depth
        self.out_N = out_N
        self.stack = nn.ModuleList()
        self.stack.append(nn.Linear(in_N, width[0]))
        for i in range(1,depth-1):
            self.stack.append(nn.Linear(width[i-1], width[i]))    
        self.stack.append(Unit1(width[depth-2], width[depth-1])) #nn.Linear

    def forward(self, x):
        for i in range(len(self.stack)):
            x = self.stack[i](x)
        return x


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)


class Unit3(nn.Module):
    def __init__(self, in_N, out_N,actf):
        super(Unit3, self).__init__()
        self.in_N = in_N
        self.out_N = out_N
        self.actf = actf
        self.L = nn.Linear(in_N, out_N)

    def forward(self, x):
        actf=self.actf
        x1 = self.L(x)
        if actf==0:
            x2 = torch.tanh(x1)
        elif actf==1:
            x2 = torch.sigmoid(x1) 
        elif actf==2:
            x2 = torch.relu(x1)
        elif actf==3:
            x2 = torch.selu(x1)
        return x2
    
class NN3(nn.Module):
    def __init__(self, in_N, width1, depth1,width2, depth2,out_N,bn,dp,dprate,actf):
        super(NN3, self).__init__()
        self.width1 = width1
        self.width2 = width2
        self.depth1 = depth1
        self.depth2 = depth2
        self.bn = bn
        self.dp = dp
        self.dprate = dprate
        self.actf = actf
        self.in_N = in_N
        self.out_N = out_N
        self.stack = nn.ModuleList()
        self.stack.append(Unit3(in_N, width1[0],actf))
        if bn==1:
            self.stack.append(nn.BatchNorm1d(width1[0]))
        for i in range(1,depth1):
            self.stack.append(Unit3(width1[i-1], width1[i],actf))
        
        if dp==1:
            self.stack.append(nn.Dropout(p=dprate))
        if depth2==1:
            self.stack.append(Unit3(width1[i], width2[0],1)) 
        else:
            self.stack.append(Unit3(width1[i], width2[0],actf))    
            for i in range(1,depth2-1):
                self.stack.append(Unit3(width2[i-1], width2[i],actf))
            self.stack.append(Unit3(width2[depth2-2], width2[depth2-1],1)) 
            
    def forward(self, x):
        for i in range(len(self.stack)):
            x = self.stack[i](x)
        return x

activation=0
dropout=0
dropout_rate=0.25341
normalization=0
batch_size=309
layers1=6
layers2=4
neurons=85
learning_rate=0.00039
L1=[neurons]*layers1
L2=[neurons]*layers2+[8]
model_h = NN3(34,L1,layers1,L2,layers2+1, 8,normalization,dropout,dropout_rate,0)

        
        
load=1
PATH="checkpoint/model-211.pt"
if load==1:
    checkpoint = torch.load(PATH)
    model_h.load_state_dict(checkpoint['model_h_state_dict'])
    optimizer2 = optim.AdamW([{'params': model_h.parameters()}], lr=learning_rate) #Default=1e-4
    optimizer2.load_state_dict(checkpoint['optimizer2_state_dict'])
    
model_h.eval()
xlo_test=np.load('xlo_test.npy')
ylo_test=np.load('ylo_test.npy')

post_analysis_test=torch.zeros([10,np.shape(xlo_test)[0],(16)])
pred_2h_star_test = model_h(torch.from_numpy(xlo_test).float())
acc_hi_0_test=torch.round(pred_2h_star_test).int()
acc_hi_1_test=torch.from_numpy(ylo_test).int()

confusion_matrix = torch.zeros(acc_hi_0_test.shape[1], 2, 2)
for i in range(acc_hi_0_test.shape[0]):
    for j in range(acc_hi_0_test.shape[1]):
        p = acc_hi_0_test[i, j]
        g = acc_hi_1_test[i, j]
        confusion_matrix[j, p, g] += 1
FP = confusion_matrix[:, 0, 1]
FN = confusion_matrix[:, 1, 0]
TP = confusion_matrix[:, 1, 1]
TN = confusion_matrix[:, 0, 0]
precision = TP / (TP + FP)
recall = TP / (TP + FN)
F1 = 2 * (precision * recall) / (precision + recall)
print('F1 for BCC phase is=',F1[1])    